In [1]:
import numpy as np
import random
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torchsurv.loss import cox
from torchvision.ops import sigmoid_focal_loss

from sklearn.metrics import roc_auc_score, root_mean_squared_error, mean_squared_error
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.util import Surv

from ncps.torch import LTC
from ncps.wirings import AutoNCP

from copy import deepcopy

RAND_SEED = 5904
random.seed(RAND_SEED)
np.random.seed(RAND_SEED)
torch.manual_seed(RAND_SEED)
torch.cuda.manual_seed(RAND_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Load Data

In [2]:
split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
train_Y = split_data["train_Y"]
test_X = split_data["test_X"]
test_Y = split_data["test_Y"]

NUM_FEATURES = train_X.shape[1]

train_surv_Y = Surv.from_arrays(train_Y[:,0], train_Y[:,1])
test_surv_Y = Surv.from_arrays(test_Y[:,0], test_Y[:,1])

## LTC LNN Model

In [3]:
def init_weights(module):
    if isinstance(module, nn.Linear):
        torch.nn.init.kaiming_normal_(module.weight, nonlinearity="leaky_relu")

        if module.bias is not None:
            nn.init.constant_(module.bias, 0)

class ProbabilityPredictorLTC(nn.Module):
    def __init__(self, input_size, num_neurons, wiring : AutoNCP, batch_first=True, return_sequences=True, ode_unfolds=6):
        super(ProbabilityPredictorLTC, self).__init__()

        self.num_neurons = num_neurons

        self.ltc_lnn = LTC(input_size=input_size,
                            units=wiring,
                            batch_first=batch_first,
                            return_sequences=return_sequences,
                            ode_unfolds=ode_unfolds)

        self.dropout1 = nn.Dropout(0.5)
        self.dropout2 = nn.Dropout(0.5)

        self.fc1 = nn.Linear(wiring.output_dim, int(wiring.output_dim * 2))
        self.normlayer1 = nn.LayerNorm(int(wiring.output_dim * 2))

        self.fc2 = nn.Linear(int(wiring.output_dim * 2), wiring.output_dim)
        self.normlayer2 = nn.LayerNorm(wiring.output_dim)

        self.fc3 = nn.Linear(wiring.output_dim, 1)

        self.leakyRelu = nn.LeakyReLU()
        self.apply(init_weights)

    def forward(self, input, timespans):

        x, _ = self.ltc_lnn(input=input, hx=None, timespans=timespans)

        # skip1 = x

        x = self.fc1(x)
        x = self.normlayer1(x)
        x = self.leakyRelu(x)
        x = self.dropout1(x)
        
        # x = x + skip1

        # skip2 = x

        x = self.fc2(x)
        x = self.normlayer2(x)
        x = self.leakyRelu(x)
        x = self.dropout2(x)

        # x = x + skip1

        output = self.fc3(x)

        return output

def get_ltc_model(num_inputs, num_outputs, num_neurons, network_sparsity=0.5, ode_unfolds=6, return_sequences=True):
    
    network_wiring = AutoNCP(num_neurons, num_outputs, sparsity_level=network_sparsity, seed=RAND_SEED)

    model = ProbabilityPredictorLTC(input_size=num_inputs,
                                    num_neurons=num_neurons,
                                    wiring=network_wiring,
                                    batch_first=True,
                                    return_sequences=return_sequences,
                                    ode_unfolds=ode_unfolds)
    
    return model

## Data

In [4]:
def get_data_sequences(features, event_times, labels, num_neurons, t_step):
    feature_sequences = list()
    time_sequences = list()
    label_sequences = list()
    sequence_masks = list()

    for feature_vector, time, label in zip(features, event_times, labels):

        divisions = time / t_step

        whole_divisions = int(divisions)
        remainder_divisions = divisions % 1

        num_time_steps = whole_divisions

        if whole_divisions > 0:
            time_seq = np.stack([t_step] * num_time_steps)
        else:
            time_seq = np.array([])

        if remainder_divisions > 0:
            time_seq = np.append(time_seq, (remainder_divisions * t_step))
            num_time_steps += 1

        assert time_seq.sum() == time

        # feature_seq = np.stack([feature_vector] * num_time_steps)
        # feature_seq = torch.tensor(feature_seq)
        # feature_sequences.append(feature_seq)

        label_seq = np.zeros_like(time_seq)
        # label_seq[-1] = label

        if label > 0:
            soften_window = 5
            abs_times = np.cumsum(time_seq)
            time_dist = time - abs_times
            
            if soften_window > len(time_seq):
                soften_window = len(time_seq)
            
            label_seq = label - (time_dist / soften_window)
            label_seq = label_seq.clip(0, 1)

            # Gaussian 
            # soften_sigma = 1.0
            # numerator = np.power(time_dist, 2)
            # denom = 2 * np.power(soften_sigma, 2)

            # label_seq = np.exp(-(numerator / denom))

        label_seq = torch.tensor(label_seq)
        label_sequences.append(label_seq)

        seq_mask = np.ones_like(time_seq)
        seq_mask = torch.tensor(seq_mask)
        sequence_masks.append(seq_mask)

        time_seq = torch.tensor(time_seq)
        time_sequences.append(time_seq)

    times_T = pad_sequence(time_sequences, batch_first=True, padding_value=1e-8, padding_side="right")
    times_T = np.expand_dims(times_T, axis=-1)
    times_T = np.broadcast_to(times_T, (times_T.shape[0], times_T.shape[1], num_neurons))
    times_T = torch.tensor(times_T, dtype=torch.float32)

    features_X = np.expand_dims(features, axis=1)
    features_X = np.repeat(features_X, times_T.shape[1], axis=1)
    features_X = torch.tensor(features_X, dtype=torch.float32)

    # features_X = pad_sequence(feature_sequences, batch_first=True, padding_value=0, padding_side="right")
    # features_X = features_X.type(torch.float32)

    labels_Y = pad_sequence(label_sequences, batch_first=True, padding_value=0, padding_side="right")
    labels_Y = np.expand_dims(labels_Y, axis=-1)
    labels_Y = torch.tensor(labels_Y, dtype=torch.float32)

    masks_M = pad_sequence(sequence_masks, batch_first=True, padding_value=0, padding_side="right")
    masks_M = np.expand_dims(masks_M, axis=-1)
    masks_M = torch.tensor(masks_M, dtype=torch.float32)

    return features_X, times_T, labels_Y, masks_M

In [5]:
class RelapseDataset(torch.utils.data.Dataset):
    def __init__(self, X, dt, Y, M):
        self.featuresX = X
        self.timesT = dt
        self.eventY = Y
        self.maskM = M

    def __len__(self):
        return len(self.featuresX)

    def __getitem__(self, idx):
        return self.featuresX[idx], self.timesT[idx], self.eventY[idx], self.maskM[idx]

def get_ltc_dataset(num_neurons, batch_size, time_step):
    # Stacking features along 2nd dimension to match the number of time steps

    train_features_X, train_times_T, train_labels_Y, train_masks_M = get_data_sequences(train_X, train_Y[:, 1], train_Y[:, 0], num_neurons, time_step)
    test_features_X, test_times_T, test_labels_Y, test_masks_M = get_data_sequences(test_X, test_Y[:, 1], test_Y[:, 0], num_neurons, time_step)

    # print(test_features_X.shape)
    # print(test_times_T.shape)
    # print(test_labels_Y.shape)

    training_data = RelapseDataset(train_features_X, train_times_T, train_labels_Y, train_masks_M)
    testing_data = RelapseDataset(test_features_X, test_times_T, test_labels_Y, test_masks_M)

    trainloader = torch.utils.data.DataLoader(training_data, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(testing_data, batch_size=batch_size, shuffle=False)

    seq_length = train_times_T.shape[1]

    return trainloader, testloader , seq_length

In [6]:
trainloader, testloader, seq_length = get_ltc_dataset(64, 64, 0.5)

In [7]:
def get_perf_metrics(model, dataloader, time_step, seq_length, device=torch.device("cpu")):
    predictions_y_hat = []
    # pred_sequences = []

    true_y = []
    time_vals = []

    # fixed_times_template = np.repeat(time_step, seq_length)

    # fixed_times_template[0] = 1.0
    # fixed_times_template[-1] -= 1.0

    # fixed_times_template = np.expand_dims(fixed_times_template, axis=(0,2))

    with torch.no_grad():
        model.eval()
        # model.ltc_lnn.return_sequences = True
        for data in dataloader:
            features, times, labels, masks = data

            features = features.to(device)
            times = times.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            output = model(input=features, timespans=times)
            output = torch.sigmoid(output)
            output = output.squeeze(-1)

            # fixed_times = np.broadcast_to(fixed_times_template, (times.shape[0], seq_length, times.shape[2]))
            # fixed_times = torch.tensor(fixed_times, dtype=torch.float32)
            # fixed_times = fixed_times.to(device)

            # fixed_output = model(input=features, timespans=fixed_times)
            # fixed_output = torch.sigmoid(fixed_output)
            # fixed_output = fixed_output.squeeze(-1)

            # pred_sequences += fixed_output.cpu().tolist()

            batch_indexes = torch.arange(0, times.shape[0])
            event_indexes = masks.sum(dim=1) - 1
            event_indexes = event_indexes.reshape(-1).tolist()
            
            preds_at_event = output[batch_indexes, event_indexes].tolist()

            hard_labels = torch.floor(labels)
            truths = hard_labels.sum(dim=1).reshape(-1).tolist()

            masked_times = times * masks
            event_times_t = masked_times.sum(dim=1).T[0].tolist()

            # # Append to predictions list
            predictions_y_hat = predictions_y_hat + preds_at_event
            true_y = true_y + truths
            time_vals = time_vals + event_times_t
    
    # pred_surv = Surv.from_arrays(np.array(true_y).astype(bool), time_vals)

    # brier_times = np.arange(time_step, (times.shape[1] * time_step) + time_step, time_step)
    # brier_times[0] += 1.0
    # brier_times[-1] -= 1.0
    
    roc = roc_auc_score(true_y, predictions_y_hat)
    rmse = root_mean_squared_error(true_y, predictions_y_hat)
    # ibs = integrated_brier_score(train_surv_Y, pred_surv, pred_sequences, brier_times)
    concordance_data = concordance_index_censored(np.array(true_y).astype(bool), time_vals, predictions_y_hat)

    print("ROC: {:.5f}".format(roc), end=" ")
    print("RMSE: {:.5f}".format(rmse), end=" ")
    # print("IBS: {:.5f}".format(ibs))
    print("C-Index: {:.5f}".format(concordance_data[0]))

    model.train()

    return concordance_data[0]

## Training Model

In [8]:
BATCH_SIZE = 64
BATCH_PRINT_STEP = 5
NUM_EPOCHS = 200
SEQ_TIME_STEP = 0.5

if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Training Over", len(train_X), "follow ups")

num_neurons = 64
num_outputs = 12

model = get_ltc_model(NUM_FEATURES, num_outputs, num_neurons, network_sparsity=0.2, return_sequences=True)
trainloader, testloader, seq_length = get_ltc_dataset(num_neurons, BATCH_SIZE, time_step=SEQ_TIME_STEP)

model = model.to(device)

total_count = len(train_Y[:, 0])

pos_count = train_Y[:, 0].sum()
neg_count = total_count - pos_count

# print(total_count, pos_count, neg_count)
pos_weight = torch.tensor(pos_count / neg_count, device=device)

bce_criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
# criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)

best_score = 0.0

# prev_bce_losses = list()
# prev_focal_losses = list()
# prev_cox_losses = list()

for epoch in range(0, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    # curr_bce_losses = list()
    # curr_focal_losses = list()
    # curr_cox_losses = list()

    for i, data in enumerate(trainloader, 0):
        features, times, labels, masks = data

        features = features.to(device)
        times = times.to(device)
        labels = labels.to(device)
        masks = masks.to(device)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward
        output = model(input=features, timespans=times)

        # BCE Loss
        bce_loss = bce_criterion(output, labels)
        masked_loss = masks * bce_loss
        bce_loss = masked_loss.sum() / masks.sum()
        raw_bce_loss = bce_loss
        
        # curr_bce_losses.append(raw_bce_loss.cpu().detach())

        # Rescale bce loss based on mean of raw bce losses from previous epoch
        # if epoch > 0:
        #     bce_loss = bce_loss / np.mean(prev_bce_losses)

        # Focal loss
        # focal_loss = sigmoid_focal_loss(output, labels, reduction="none")
        # masked_loss = masks * focal_loss
        # focal_loss = masked_loss.sum() / masks.sum()
        # raw_focal_loss = focal_loss

        # curr_focal_losses.append(raw_focal_loss.cpu().detach())

        # Rescale bce loss based on mean of raw bce losses from previous epoch
        # if epoch > 0:
        #     focal_loss = focal_loss / np.mean(prev_focal_losses)

        # Cox Loss
        output = output.squeeze(-1)

        batch_indexes = torch.arange(0, times.shape[0])
        event_indexes = masks.sum(dim=1) - 1
        event_indexes = event_indexes.reshape(-1).tolist()
        
        preds_at_event = output[batch_indexes, event_indexes]

        truths = labels.sum(dim=1).reshape(-1)
        event_times_t = times.sum(dim=1).T[0]

        cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths.type(torch.bool), event_times_t)
        raw_cox_loss = cox_loss

        # curr_cox_losses.append(raw_cox_loss.cpu().detach())

        # Rescale cox loss based on mean of raw cox losses from previous epoch
        # if epoch > 0:
        #     cox_loss = cox_loss / np.mean(prev_cox_losses)

        # Backward
        loss = cox_loss + (0.1 * bce_loss)
        raw_loss = raw_cox_loss + raw_bce_loss
        
        loss.backward()
        
        # Optimize
        optimizer.step()

        running_loss += loss.item()

        if i % BATCH_PRINT_STEP == (BATCH_PRINT_STEP - 1):
            curr_loss = running_loss / BATCH_PRINT_STEP
            epochBatchLossPrint = "Epoch: {} Batch: {} Cox Loss: {:.5f} BCE Loss: {:.5f} Joint Loss: {:.5f} Raw Loss: {:.5f}".format(epoch + 1, i + 1, cox_loss, bce_loss, curr_loss, raw_loss)
            print(epochBatchLossPrint)
            running_loss = 0.0

    # prev_bce_losses = deepcopy(curr_bce_losses)
    # prev_cox_losses = deepcopy(curr_cox_losses)
    # prev_focal_losses = deepcopy(curr_focal_losses)

    # Testing Loss
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for data in testloader:
            features, times, labels, masks = data

            features = features.to(device)
            times = times.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            output = model(input=features, timespans=times)
            
            # BCE Loss
            bce_loss = bce_criterion(output, labels)
            masked_loss = masks * bce_loss
            bce_loss = masked_loss.sum() / masks.sum()

            # Focal loss
            # focal_loss = sigmoid_focal_loss(output, labels, reduction="none")
            # masked_loss = masks * focal_loss
            # focal_loss = masked_loss.sum() / masks.sum()

            # Cox Loss
            output = output.squeeze(-1)

            batch_indexes = torch.arange(0, times.shape[0])
            event_indexes = masks.sum(dim=1) - 1
            event_indexes = event_indexes.reshape(-1).tolist()
            
            preds_at_event = output[batch_indexes, event_indexes]

            truths = labels.sum(dim=1).reshape(-1)
            event_times_t = times.sum(dim=1).T[0]

            cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths.type(torch.bool), event_times_t)

            # Joint Loss
            loss = cox_loss + (0.1 * bce_loss)

            val_loss += loss.item() * BATCH_SIZE
            
        val_loss /= len(testloader.dataset)

    print("Epoch: {} Testing Loss: {:.5f}".format(epoch + 1, val_loss))

    print("Training Data:")
    cindex_train = get_perf_metrics(model=model, dataloader=trainloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)
    print("Testing Data:")
    cindex_test = get_perf_metrics(model=model, dataloader=testloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)

    curr_score = cindex_train + cindex_test
    
    save_objective = cindex_test - (0.75 * abs(cindex_train - cindex_test))

    if(save_objective > best_score):
        best_score = save_objective
        print("New Model Weights Saved")
        torch.save(model.state_dict(), "weights_LTC.pth")

Device: cuda
Training Over 704 follow ups
Epoch: 1 Batch: 5 Cox Loss: 4.12332 BCE Loss: 1.61737 Joint Loss: 4.99415 Raw Loss: 5.74069
Epoch: 1 Batch: 10 Cox Loss: 4.74179 BCE Loss: 1.63996 Joint Loss: 5.55456 Raw Loss: 6.38175
Epoch: 1 Testing Loss: 4.10925
Training Data:
ROC: 0.48615 RMSE: 0.78263 C-Index: 0.46896
Testing Data:
ROC: 0.52478 RMSE: 0.78053 C-Index: 0.53256
New Model Weights Saved
Epoch: 2 Batch: 5 Cox Loss: 5.18221 BCE Loss: 1.66497 Joint Loss: 4.60736 Raw Loss: 6.84718
Epoch: 2 Batch: 10 Cox Loss: 4.71449 BCE Loss: 1.63718 Joint Loss: 4.72871 Raw Loss: 6.35167
Epoch: 2 Testing Loss: 4.09100
Training Data:
ROC: 0.52304 RMSE: 0.76376 C-Index: 0.50314
Testing Data:
ROC: 0.55050 RMSE: 0.76146 C-Index: 0.56036
New Model Weights Saved


C:\Users\alvin\AppData\Local\Temp\ipykernel_15664\2574669820.py:98: UserWarning: Ties in `time` detected; using efron method to handle ties.
  cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths.type(torch.bool), event_times_t)


Epoch: 3 Batch: 5 Cox Loss: 4.77021 BCE Loss: 1.62733 Joint Loss: 4.91693 Raw Loss: 6.39754
Epoch: 3 Batch: 10 Cox Loss: 4.43785 BCE Loss: 1.58973 Joint Loss: 4.87206 Raw Loss: 6.02758
Epoch: 3 Testing Loss: 4.07915
Training Data:
ROC: 0.57293 RMSE: 0.74926 C-Index: 0.55752
Testing Data:
ROC: 0.54709 RMSE: 0.74728 C-Index: 0.56817
New Model Weights Saved
Epoch: 4 Batch: 5 Cox Loss: 4.29923 BCE Loss: 1.56843 Joint Loss: 4.49042 Raw Loss: 5.86766
Epoch: 4 Batch: 10 Cox Loss: 4.06911 BCE Loss: 1.57125 Joint Loss: 4.32491 Raw Loss: 5.64035
Epoch: 4 Testing Loss: 4.06510
Training Data:
ROC: 0.56936 RMSE: 0.73083 C-Index: 0.56933
Testing Data:
ROC: 0.56066 RMSE: 0.72897 C-Index: 0.58298
New Model Weights Saved
Epoch: 5 Batch: 5 Cox Loss: 4.45636 BCE Loss: 1.53699 Joint Loss: 4.34749 Raw Loss: 5.99335


C:\Users\alvin\AppData\Local\Temp\ipykernel_15664\2574669820.py:98: UserWarning: Ties in `time` detected; using efron method to handle ties.
  cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths.type(torch.bool), event_times_t)


Epoch: 5 Batch: 10 Cox Loss: 4.39743 BCE Loss: 1.49460 Joint Loss: 4.48256 Raw Loss: 5.89203
Epoch: 5 Testing Loss: 4.04814
Training Data:
ROC: 0.56836 RMSE: 0.70307 C-Index: 0.57307
Testing Data:
ROC: 0.56566 RMSE: 0.70127 C-Index: 0.58752
New Model Weights Saved
Epoch: 6 Batch: 5 Cox Loss: 4.52312 BCE Loss: 1.45940 Joint Loss: 4.48833 Raw Loss: 5.98252
Epoch: 6 Batch: 10 Cox Loss: 4.13694 BCE Loss: 1.41668 Joint Loss: 4.17599 Raw Loss: 5.55362
Epoch: 6 Testing Loss: 4.03411
Training Data:
ROC: 0.54348 RMSE: 0.66823 C-Index: 0.55285
Testing Data:
ROC: 0.56437 RMSE: 0.66665 C-Index: 0.58906
Epoch: 7 Batch: 5 Cox Loss: 4.13229 BCE Loss: 1.36325 Joint Loss: 4.09160 Raw Loss: 5.49554
Epoch: 7 Batch: 10 Cox Loss: 4.16514 BCE Loss: 1.34545 Joint Loss: 4.16659 Raw Loss: 5.51059
Epoch: 7 Testing Loss: 4.02927
Training Data:
ROC: 0.52332 RMSE: 0.63955 C-Index: 0.53617
Testing Data:
ROC: 0.57233 RMSE: 0.63859 C-Index: 0.58588
Epoch: 8 Batch: 5 Cox Loss: 3.80908 BCE Loss: 1.30276 Joint Loss: 4.0

...

Epoch: 193 Batch: 10 Cox Loss: 3.35942 BCE Loss: 0.21072 Joint Loss: 3.46743 Raw Loss: 3.57014
Epoch: 193 Testing Loss: 3.60383
Training Data:
ROC: 0.78884 RMSE: 0.34501 C-Index: 0.76550
Testing Data:
ROC: 0.72562 RMSE: 0.36168 C-Index: 0.70606
Epoch: 194 Batch: 5 Cox Loss: 3.18949 BCE Loss: 0.18472 Joint Loss: 3.49560 Raw Loss: 3.37420
Epoch: 194 Batch: 10 Cox Loss: 3.27938 BCE Loss: 0.14563 Joint Loss: 3.57505 Raw Loss: 3.42501
Epoch: 194 Testing Loss: 3.67753
Training Data:
ROC: 0.78415 RMSE: 0.35358 C-Index: 0.76460
Testing Data:
ROC: 0.68834 RMSE: 0.37202 C-Index: 0.68280
Epoch: 195 Batch: 5 Cox Loss: 3.13438 BCE Loss: 0.15535 Joint Loss: 3.38745 Raw Loss: 3.28973
Epoch: 195 Batch: 10 Cox Loss: 3.09764 BCE Loss: 0.16531 Joint Loss: 3.46449 Raw Loss: 3.26294
Epoch: 195 Testing Loss: 3.60220
Training Data:
ROC: 0.79386 RMSE: 0.34472 C-Index: 0.77039
Testing Data:
ROC: 0.72293 RMSE: 0.36293 C-Index: 0.70420
Epoch: 196 Batch: 5 Cox Loss: 3.53711 BCE Loss: 0.17806 Joint Loss: 3.33150 R

C:\Users\alvin\AppData\Local\Temp\ipykernel_15664\2574669820.py:98: UserWarning: Ties in `time` detected; using efron method to handle ties.
  cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths.type(torch.bool), event_times_t)


Epoch: 198 Batch: 10 Cox Loss: 3.26018 BCE Loss: 0.13327 Joint Loss: 3.43228 Raw Loss: 3.39346
Epoch: 198 Testing Loss: 3.64413
Training Data:
ROC: 0.79501 RMSE: 0.34578 C-Index: 0.77096
Testing Data:
ROC: 0.70785 RMSE: 0.36642 C-Index: 0.69929


C:\Users\alvin\AppData\Local\Temp\ipykernel_15664\2574669820.py:98: UserWarning: Ties in `time` detected; using efron method to handle ties.
  cox_loss = cox.neg_partial_log_likelihood(preds_at_event, truths.type(torch.bool), event_times_t)


Epoch: 199 Batch: 5 Cox Loss: 3.53311 BCE Loss: 0.17828 Joint Loss: 3.52560 Raw Loss: 3.71140
Epoch: 199 Batch: 10 Cox Loss: 3.39206 BCE Loss: 0.17569 Joint Loss: 3.49212 Raw Loss: 3.56775
Epoch: 199 Testing Loss: 3.64317
Training Data:
ROC: 0.79570 RMSE: 0.34476 C-Index: 0.77315
Testing Data:
ROC: 0.70584 RMSE: 0.36590 C-Index: 0.69761
Epoch: 200 Batch: 5 Cox Loss: 3.43812 BCE Loss: 0.14465 Joint Loss: 3.52802 Raw Loss: 3.58277
Epoch: 200 Batch: 10 Cox Loss: 3.65617 BCE Loss: 0.12734 Joint Loss: 3.59650 Raw Loss: 3.78351
Epoch: 200 Testing Loss: 3.65808
Training Data:
ROC: 0.78983 RMSE: 0.35058 C-Index: 0.77269
Testing Data:
ROC: 0.70300 RMSE: 0.37098 C-Index: 0.69702


## Metrics

In [11]:
if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

model = model.to(device)

print("Training Data:")
get_perf_metrics(model=model, dataloader=trainloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)
print("Testing Data:")
get_perf_metrics(model=model, dataloader=testloader, time_step=SEQ_TIME_STEP, seq_length=seq_length, device=device)

Device: cuda
Training Data:
ROC: 0.72840 RMSE: 0.36846 C-Index: 0.71856
Testing Data:
ROC: 0.72543 RMSE: 0.37060 C-Index: 0.71555


np.float64(0.715550912889454)

In [ ]:
state_dict = torch.load("../models/weights_LTC.pth", weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [ ]:
months = 24
num_steps = 48

time_diff = float(months) / float(num_steps)

# print(train_features_X[0].shape)
# print(train_times_T[0].shape)

# Convert a single test case to the right shape [Batch, Time, Neuron]
# In the ode solver, the time is used for calculations in each neuron.
# The calculations are performed elementwise so 
# Create list of time points

timespans_list = [time_diff] * num_steps

# Convert to numpy array
single_test_t = np.stack(timespans_list)

single_test_t = np.expand_dims(single_test_t, axis=-1)
single_test_t = np.broadcast_to(single_test_t, (single_test_t.shape[0], num_neurons))
single_test_t = torch.tensor(single_test_t).float().unsqueeze(0)
# print(single_test_t.shape)

# print(test_Y[:,0][test_index])

# Convert a single test case to the right shape [Batch, Vector, Feature]
test_index = 4
single_feature_vector = test_X[test_index]
print("label:", test_Y[:, 0][test_index])
print("time:",test_Y[:, 1][test_index])
# Make copies of the feature vector to match the number of prediction time points
copied_vectors = [single_feature_vector] * len(timespans_list)

single_test_X = np.stack(copied_vectors, axis=0)
# print(single_test_X.shape)
single_test_X = torch.tensor(single_test_X).float().unsqueeze(0)
# print(single_test_X)

with torch.no_grad():
    model.eval()
    model.to("cpu")

    model.ltc_lnn.return_sequences = True
    pred = model(input=single_test_X, timespans=single_test_t)
    pred = torch.sigmoid(pred)
    
pred_flatten = pred.reshape(-1).numpy()

# print(pred_flatten.shape)
print(pred_flatten)

In [ ]:
x_values = np.arange(time_diff, months+time_diff, time_diff)

pred_rescaled = np.multiply(pred_flatten, time_diff)
cumulative_hazard = np.cumsum(pred_rescaled)
diffs = np.diff(pred_flatten, prepend=True)

surv_func = np.exp(-cumulative_hazard)
# surv_func = np.cumprod(1 - pred_rescaled)

plt.figure()
plt.step(x_values, surv_func, where="post")
plt.xlim([0, months])
# plt.ylim([0, 1])
# plt.plot(x_values, surv_func)
plt.show()